# Creating Videos
This notebook covers how to use the headset_localization package to create video visualizations of the localizers

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

from headset_localization import *
from shared import CompleteRobotScan

In [ ]:
round1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")

robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

labeled_headset_data = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec1")

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

### PnP localizer video generation
Show matching with 1 ref image

In [ ]:
pnp_video_predictor = PnPLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(display_matching=False, crop_augmentations=[0.4]),
)

init_predictor_grade = PredictionOnDataset(
    predictor = pnp_video_predictor,
    headset_data = labeled_headset_data,
    vid_gen=VideoGenerator(fps=5, 
        style_config=FeatureStyleConfig(
            connection_line_alpha=0.4, 
            point_size=6, 
            connection_line_thickness=2, 
            show_info_card=True, 
            points_3d_size = 0.006,
            overlay_font_size=20
        ),
        scan_bgr_images=robot_env.robot_bgr_images[0:1],
        scan_xyz_images=robot_env.robot_xyz_images[0:1],
        use_second_3d_axis=True,
        figsize_3d=(labeled_headset_data.bgr_image_s.shape[2], labeled_headset_data.bgr_image_s.shape[1])
    ),
    video_save_location="test_pnp_lai_2.mp4"
)
init_predictor_grade.print_summary()

### PnP+L localizer video generation

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan


mixed1_scan = CompleteRobotScan.from_folder("../example_datasets/mixed1/mixed1_scan")

mixed1_env = Scanned3dEnvironment.from_gathered_robot_data(
        mixed1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

#mixed1_rec1 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec1")
mixed1_rec2 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec2")



mixed1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")

mixed1_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

mixed1_rec2 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec1")




For matching

In [ ]:
pnpl_video_predictor = PnPLLocalizer(
        cam2_intrinsic_mtx=mixed1_rec2.intrinsic_cam_mtx,
        cam1_bgr_images=mixed1_env.robot_bgr_images,
        cam1_xyz_images=mixed1_env.robot_xyz_images,
        cam1_line_generator = LineGenerator(visualize_cleanup=False),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4]
        ),
        cam2_line_generator=LineGenerator(
            line_cleanup_config= MultiPassLineMergingConfig(passes=[
            LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850),
            LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 20/850)
            ]),
            visualize_cleanup=False
        ),
        line_fitting_3d_config = LineFitting3dConfig(fitting_algorithm='pca', min_number_points=4),
        line_matching_config = LineMatchingConfig(max_point_line_dist_px=20, better_factor=1.2),
        debug_visualize_matching = False,
        debug_visualize_3d = False
)

init_predictor_grade = PredictionOnDataset(
    predictor = pnpl_video_predictor,
    headset_data = mixed1_rec2,
    vid_gen=VideoGenerator(fps=5, 
        style_config=FeatureStyleConfig(connection_line_alpha=0.0, point_size=5, line_widht=4, point_alpha = 0.4, show_3d_points=False, show_info_card=False),
        scan_bgr_images=mixed1_env.robot_bgr_images,
        scan_xyz_images=mixed1_env.robot_xyz_images,
        use_second_3d_axis=False,
        figsize_3d=(mixed1_rec2.bgr_image_s.shape[2], mixed1_rec2.bgr_image_s.shape[1])
    ),
    video_save_location="test_pnpl_lai_formatching.mp4"
)
init_predictor_grade.print_summary()

For Video

In [ ]:
pnpl_video_predictor = PnPLLocalizer(
        cam2_intrinsic_mtx=mixed1_rec2.intrinsic_cam_mtx,
        cam1_bgr_images=mixed1_env.robot_bgr_images[0:1],
        cam1_xyz_images=mixed1_env.robot_xyz_images[0:1],
        cam1_line_generator = LineGenerator(visualize_cleanup=False),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4]
        ),
        cam2_line_generator=LineGenerator(
            line_cleanup_config= MultiPassLineMergingConfig(passes=[
            LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850),
            LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 20/850)
            ]),
            visualize_cleanup=False
        ),
        line_fitting_3d_config = LineFitting3dConfig(fitting_algorithm='pca', min_number_points=4),
        line_matching_config = LineMatchingConfig(max_point_line_dist_px=20, better_factor=1.2),
        debug_visualize_matching = False,
        debug_visualize_3d = False
)

init_predictor_grade = PredictionOnDataset(
    predictor = pnpl_video_predictor,
    headset_data = mixed1_rec2,
    vid_gen=VideoGenerator(fps=5, 
        style_config=FeatureStyleConfig(connection_line_alpha=0.0, point_size=4, line_widht=4, show_3d_points=True,points_3d_size=0.004, show_info_card=True, overlay_font_size=20),
        scan_bgr_images=mixed1_env.robot_bgr_images[0:1],
        scan_xyz_images=mixed1_env.robot_xyz_images[0:1],
        use_second_3d_axis=True,
        figsize_3d=(mixed1_rec2.bgr_image_s.shape[2], mixed1_rec2.bgr_image_s.shape[1])
    ),
    video_save_location="test_pnpl_lai_2.mp4"
)
init_predictor_grade.print_summary()

### Ellipsoid localizer video generation

In [ ]:
# mixed1_scan = CompleteRobotScan.from_folder("../example_datasets/mixed1/mixed1_scan")

# mixed1_env = Scanned3dEnvironment.from_gathered_robot_data(
#         mixed1_scan, number_of_sampled_datapoints=100,
#         est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
#         est3d_xyz_icp_config=ICPAlignmentConfig()
# )

# mixed1_rec1 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec1")

mixed1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")

mixed1_env = Scanned3dEnvironment.from_gathered_robot_data(
        mixed1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

mixed1_rec1 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec1")

In [ ]:

yolo = YOLOv26Segmenter(
    "yoloe-26l-seg.pt", 
    prompts=[
        "cup", "fruit", "plate", "teddy", "ball", "tennisball", "pen", "lego", "brick", "duplo", 
        "pen", "sphere", "round object", "tool", "toy", "plastic object", "bowl", "object", "metal" 
    ]
)

ellipsoid_video_predictor = EllipsoidLocalizer(
        cam2_intrinsic_mtx=mixed1_rec1.intrinsic_cam_mtx,
        cam1_bgr_images=mixed1_env.robot_bgr_images,
        cam1_xyz_images=mixed1_env.robot_xyz_images,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = yolo,
        cam2_segmenter=yolo,
        ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(contamination=0.02, size_penalty=0.05, size_p_norm=4),            
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2, max_color_dist=10),
)

ellipsoid_predictor_grade = PredictionOnDataset(
    predictor = ellipsoid_video_predictor,
    headset_data = mixed1_rec1,
    vid_gen=VideoGenerator(
        fps=5, 
        style_config=FeatureStyleConfig(
            overlay_font_size = 20, unmatched_alpha=0.0, connection_line_alpha = 0.0, line_widht = 2, show_3d_points=False, point_size=0
        ),
        scan_bgr_images=mixed1_env.robot_bgr_images,
        scan_xyz_images=mixed1_env.robot_xyz_images,
        use_second_3d_axis=True,
        figsize_3d=(mixed1_rec1.bgr_image_s.shape[2], mixed1_rec1.bgr_image_s.shape[1])
    ),
    video_save_location="test_ellipsoid_lai2.mp4"
)
ellipsoid_predictor_grade.print_summary()

## Gaze Intersection Error Visualization

In [ ]:
gie_robot_env = Scanned3dEnvironment.from_gathered_robot_data(
         CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan"), number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)
gie_headset_data = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec1")

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

gaze_predictor_grade = PredictionOnDataset(
    predictor = PnPLocalizer(
        cam2_intrinsic_mtx=gie_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=gie_robot_env.robot_bgr_images,
        cam1_xyz_images=gie_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(display_matching=False, crop_augmentations=[0.4]),
    ),
    headset_data = gie_headset_data,
)

gaze_error_calculator = FastRayIntersectionError(
    points=gie_robot_env.robot_xyz_images, 
    intrinsics=gie_headset_data.intrinsic_cam_mtx
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import open3d as o3d

from shared import create_3d_camera

GAZE_LOCATION = (gie_headset_data.bgr_image_s.shape[2]//2, int(gie_headset_data.bgr_image_s.shape[1]//2*1.2))


def get_intersection_sphere_list(pose:np.ndarray, color:np.ndarray | list[float], radius = 0.01):
    gt_intersect = gaze_error_calculator._pixels_to_hitpoints(
        base_t_cam=pose, pixels=np.array([[GAZE_LOCATION[0], GAZE_LOCATION[1]]])
    )
    if gt_intersect.shape[0] == 0:
        return []

    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere.paint_uniform_color(color)
    sphere.translate(gt_intersect[0])
    return [sphere]

o3d_vis = o3d.visualization.Visualizer()
o3d_vis.create_window(
    window_name='Open3D', 
    width= gie_headset_data.bgr_image_s.shape[2], 
    height= gie_headset_data.bgr_image_s.shape[1], 
    visible = False
)
o3d_vis.add_geometry(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.2))



def set_camera3d_viewpoint(base_t_cam:np.ndarray):  
    view_control = o3d_vis.get_view_control()
    camera_params = view_control.convert_to_pinhole_camera_parameters()
    camera_params.extrinsic = np.linalg.inv(base_t_cam)
    K = gie_headset_data.intrinsic_cam_mtx
    w, h = gie_headset_data.bgr_image_s.shape[2], gie_headset_data.bgr_image_s.shape[1]
    camera_params.intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, K[0, 0], K[1, 1], K[0, 2], K[1, 2])
    view_control.convert_from_pinhole_camera_parameters(camera_params)  


world_points = gie_robot_env.robot_xyz_images.reshape(-1,3)
no_nan_mask = np.isfinite(world_points).all(axis=-1)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(world_points[no_nan_mask])
world_colors = gie_robot_env.robot_bgr_images.reshape(-1,3).astype(np.float32)[:, ::-1]/255
pcd.colors = o3d.utility.Vector3dVector(world_colors[no_nan_mask])
o3d_vis.add_geometry(pcd)

frames = []

for INDEX, headset_img in enumerate(gie_headset_data.bgr_image_s):
    pose = [(pred, gt) for i, pred, gt in gaze_predictor_grade.comparable_poses if i == INDEX]
    if len(pose) < 1:
        continue

    pred_pose, gt_pose = pose[0]

    # Show headset POV
    headset_img_with_dot = headset_img.copy()
    cv2.circle(headset_img_with_dot, 
           (GAZE_LOCATION[0], GAZE_LOCATION[1]), 
           radius=6,
           color=(0, 255, 0),
           thickness=-1
        )
    headset_img_with_dot = cv2.cvtColor(headset_img_with_dot, cv2.COLOR_RGB2BGR)

    
    # Show GIE for one point 3D
    to_vis = []

    to_vis += get_intersection_sphere_list(pose=gt_pose, color=[0.0, 1.0, 0.0])
    to_vis += get_intersection_sphere_list(pose=pred_pose, color=[1.0, 0.0, 0.0])

    for geom in to_vis:
        o3d_vis.add_geometry(geom)

    set_camera3d_viewpoint(base_t_cam=pred_pose)

    o3d_vis.poll_events()
    o3d_vis.update_renderer()
    image_array_3d = o3d_vis.capture_screen_float_buffer(do_render=True)
    image_array_3d = (np.asarray(image_array_3d) * 255).astype(np.uint8)

    for geom in to_vis:
        o3d_vis.remove_geometry(geom)
    to_vis.clear()

    frames.append(np.hstack([headset_img_with_dot, image_array_3d]))
    

height, width = frames[0].shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("GIE_video.mp4", fourcc, 5, (width, height))

for frame in frames:
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    out.write(frame_bgr)
out.release()